# Databases and the Relational Model

Notebook for **Chapter 1** of the *sql* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/sql/ch01.html

Tip: change a number, predict the output, then run the cell again.

## 1.1 Why Databases Exist

In [ ]:
orders = [{"status": "paid", "amount": 500}, {"status": "cancelled", "amount": 300}]
total = 0
for order in orders:
    if order["status"] == "paid":
        total += order["amount"]

## 1.2 Tables, Rows, Columns and Keys

## 1.3 Setting Up SQLite and PostgreSQL

## 1.4 Relational Thinking: Relations, Cardinality and Why We Split Tables

## 1.5 Types and Constraints: Teaching the Database to Refuse Bad Data

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("""
CREATE TABLE products (
  product_id INTEGER PRIMARY KEY,
  name       TEXT    NOT NULL,
  sku        TEXT    UNIQUE,
  price      INTEGER NOT NULL CHECK (price > 0),
  category   TEXT    NOT NULL DEFAULT 'General'
)""")
tests = [
 ("good row",     "INSERT INTO products(product_id,name,sku,price) VALUES (1,'Keyboard','KB-1',1500)"),
 ("missing name", "INSERT INTO products(product_id,sku,price) VALUES (2,'MS-1',600)"),
 ("duplicate sku","INSERT INTO products(product_id,name,sku,price) VALUES (3,'Mouse','KB-1',600)"),
 ("price zero",   "INSERT INTO products(product_id,name,sku,price) VALUES (4,'Free','F-1',0)"),
 ("duplicate id", "INSERT INTO products(product_id,name,sku,price) VALUES (1,'Pen','P-1',20)"),
]
for label, sql in tests:
    try:
        con.execute(sql)
        print(f"{label:14} OK")
    except sqlite3.Error as e:
        print(f"{label:14} REJECTED: {e}")
print(con.execute("SELECT * FROM products").fetchall())

good row       OK
missing name   REJECTED: NOT NULL constraint failed: products.name
duplicate sku  REJECTED: UNIQUE constraint failed: products.sku
price zero     REJECTED: CHECK constraint failed: price > 0
duplicate id   REJECTED: UNIQUE constraint failed: products.product_id
[(1, 'Keyboard', 'KB-1', 1500, 'General')]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE lines_a (qty INTEGER CHECK (qty > 0))")
con.execute("CREATE TABLE lines_b (qty INTEGER NOT NULL CHECK (qty > 0))")
for t in ("lines_a", "lines_b"):
    for value in (5, 0, None):
        try:
            con.execute(f"INSERT INTO {t} VALUES (?)", (value,))
            print(t, value, "accepted")
        except sqlite3.Error as e:
            print(t, value, "rejected:", e)

lines_a 5 accepted
lines_a 0 rejected: CHECK constraint failed: qty > 0
lines_a None accepted
lines_b 5 accepted
lines_b 0 rejected: CHECK constraint failed: qty > 0
lines_b None rejected: NOT NULL constraint failed: lines_b.qty


In [ ]:
import sqlite3
from decimal import Decimal

print(0.1 + 0.2, 0.1 + 0.2 == 0.3)
total = 0.0
for _ in range(10):
    total += 0.1
print(total, total == 1.0)

con = sqlite3.connect(":memory:")
print(con.execute("SELECT 0.1 + 0.2 = 0.3, 10 + 20 = 30").fetchone())

print(Decimal("0.1") + Decimal("0.2"))
paise = [10] * 10
print(sum(paise), sum(paise) / 100)

0.30000000000000004 False
0.9999999999999999 False
(0, 1)
0.3
100 1.0


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE loose (n INTEGER)")
con.execute("INSERT INTO loose VALUES ('42'), ('abc'), (3.0), (3.5)")
print(con.execute("SELECT n, typeof(n) FROM loose").fetchall())

con.execute("CREATE TABLE tight (n INTEGER) STRICT")
try:
    con.execute("INSERT INTO tight VALUES ('abc')")
except sqlite3.Error as e:
    print("STRICT says:", e)

[(42, 'integer'), ('abc', 'text'), (3, 'integer'), (3.5, 'real')]
STRICT says: cannot store TEXT value in INTEGER column tight.n


## 1.6 Keys in Depth: Surrogate, Composite, Self-Referencing and What Happens on Delete

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE a (id INTEGER PRIMARY KEY, v TEXT)")
con.execute("CREATE TABLE b (id INTEGER PRIMARY KEY AUTOINCREMENT, v TEXT)")
for t in ("a", "b"):
    for v in ("x", "y", "z"):
        con.execute(f"INSERT INTO {t}(v) VALUES (?)", (v,))
    con.execute(f"DELETE FROM {t} WHERE id = 3")
    cur = con.execute(f"INSERT INTO {t}(v) VALUES ('new')")
    print(t, "new row got id", cur.lastrowid)

a new row got id 3
b new row got id 4


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE students (student_id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE courses  (course_id  INTEGER PRIMARY KEY, title TEXT NOT NULL);
CREATE TABLE enrolments (
  student_id INTEGER NOT NULL REFERENCES students(student_id),
  course_id  INTEGER NOT NULL REFERENCES courses(course_id),
  grade      INTEGER,
  PRIMARY KEY (student_id, course_id)
);
INSERT INTO students VALUES (1,'Asha'),(2,'Ravi');
INSERT INTO courses  VALUES (10,'SQL'),(20,'Python');
""")
attempts = [(1,10), (1,20), (1,10), (2,10), (3,10)]
for s, c in attempts:
    try:
        con.execute("INSERT INTO enrolments(student_id, course_id) VALUES (?,?)", (s, c))
        print((s, c), "enrolled")
    except sqlite3.Error as e:
        print((s, c), "refused:", e)

(1, 10) enrolled
(1, 20) enrolled
(1, 10) refused: UNIQUE constraint failed: enrolments.student_id, enrolments.course_id
(2, 10) enrolled
(3, 10) refused: FOREIGN KEY constraint failed


In [ ]:
import sqlite3

def trial(action):
    con = sqlite3.connect(":memory:")
    con.execute("PRAGMA foreign_keys = ON")
    con.executescript(f"""
    CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT);
    CREATE TABLE employees (
      emp_id INTEGER PRIMARY KEY, name TEXT,
      dept_id INTEGER REFERENCES departments(dept_id) {action});
    INSERT INTO departments VALUES (10,'Testing'),(20,'Data');
    INSERT INTO employees VALUES (1,'Asha',10),(2,'Meena',10),(3,'Ravi',20);
    """)
    try:
        con.execute("DELETE FROM departments WHERE dept_id = 10")
        result = "delete ok"
    except sqlite3.Error as e:
        result = "delete refused: " + str(e)
    rows = con.execute("SELECT name, dept_id FROM employees ORDER BY emp_id").fetchall()
    return result, rows

for action in ("", "ON DELETE CASCADE", "ON DELETE SET NULL"):
    print(repr(action or "default"), trial(action))

'default' ('delete refused: FOREIGN KEY constraint failed', [('Asha', 10), ('Meena', 10), ('Ravi', 20)])
'ON DELETE CASCADE' ('delete ok', [('Ravi', 20)])
'ON DELETE SET NULL' ('delete ok', [('Asha', None), ('Meena', None), ('Ravi', 20)])


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE employees (
  emp_id     INTEGER PRIMARY KEY,
  name       TEXT NOT NULL,
  manager_id INTEGER REFERENCES employees(emp_id)
);
INSERT INTO employees VALUES (1,'Kavya',NULL),(2,'Asha',1),(3,'Ravi',1),(4,'Meena',2);
""")
q = """SELECT e.name, m.name AS manager
       FROM employees e LEFT JOIN employees m ON m.emp_id = e.manager_id
       ORDER BY e.emp_id"""
for row in con.execute(q):
    print(row)

('Kavya', None)
('Asha', 'Kavya')
('Ravi', 'Kavya')
('Meena', 'Asha')


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE order_items (order_id INT, product_id INT, qty INT);
INSERT INTO order_items VALUES (101,1,1),(101,3,2),(102,2,1),(103,5,1),(103,4,1),(104,3,5);
""")
print(con.execute("SELECT COUNT(*), COUNT(DISTINCT order_id) FROM order_items").fetchone())
print(con.execute("""SELECT COUNT(*) FROM
                     (SELECT DISTINCT order_id, product_id FROM order_items)""").fetchone())

(6, 4)
(6,)


## 1.7 The Relational Model Itself: Relations, Sets and Three Core Operations

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT NOT NULL);
CREATE TABLE employees (emp_id INTEGER PRIMARY KEY, name TEXT, dept_id INTEGER, salary INTEGER);
INSERT INTO departments VALUES (10,'Testing'),(20,'Data');
INSERT INTO employees VALUES (1,'Asha',10,90000),(2,'Ravi',20,120000),(3,'Meena',10,95000);
""")
print("selection :", con.execute("SELECT name FROM employees WHERE salary >= 95000").fetchall())
print("projection:", con.execute("SELECT DISTINCT dept_id FROM employees").fetchall())
print("join      :", con.execute("""SELECT e.name, d.dept_name
                                    FROM employees e JOIN departments d ON d.dept_id = e.dept_id
                                    ORDER BY e.emp_id""").fetchall())

selection : [('Ravi',), ('Meena',)]
projection: [(10,), (20,)]
join      : [('Asha', 'Testing'), ('Ravi', 'Data'), ('Meena', 'Testing')]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT);
CREATE TABLE employees (emp_id INTEGER PRIMARY KEY, name TEXT, dept_id INTEGER);
INSERT INTO departments VALUES (10,'Testing'),(20,'Data');
INSERT INTO employees VALUES (1,'Asha',10),(2,'Ravi',20),(3,'Meena',10);
""")
product = con.execute("SELECT COUNT(*) FROM employees e CROSS JOIN departments d").fetchone()[0]
matched = con.execute("""SELECT COUNT(*) FROM employees e CROSS JOIN departments d
                         WHERE d.dept_id = e.dept_id""").fetchone()[0]
joined  = con.execute("""SELECT COUNT(*) FROM employees e
                         JOIN departments d ON d.dept_id = e.dept_id""").fetchone()[0]
print(product, matched, joined)

6 3 3


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, city TEXT, note TEXT)")
con.executemany("INSERT INTO t(city, note) VALUES (?, ?)",
                [("Pune", "a" * 40), ("Delhi", "b" * 40), ("Mumbai", "c" * 40), ("Chennai", "d" * 40)])
print("before index:", [r[0] for r in con.execute("SELECT city FROM t")])
con.execute("CREATE INDEX t_city ON t(city)")
print("after index :", [r[0] for r in con.execute("SELECT city FROM t")])
print("with ORDER BY:", [r[0] for r in con.execute("SELECT city FROM t ORDER BY city")])

before index: ['Pune', 'Delhi', 'Mumbai', 'Chennai']
after index : ['Chennai', 'Delhi', 'Mumbai', 'Pune']
with ORDER BY: ['Chennai', 'Delhi', 'Mumbai', 'Pune']


## 1.8 Why We Split Tables: Anomalies, Atomic Values and a Lossless Split

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE sales (
  order_id INTEGER PRIMARY KEY, customer_name TEXT NOT NULL, customer_city TEXT,
  customer_email TEXT, product_name TEXT NOT NULL, price INTEGER NOT NULL);
INSERT INTO sales VALUES
 (101,'Asha','Pune','asha@x.com','Keyboard',1500),
 (103,'Asha','Pune','asha@x.com','Desk Lamp',900),
 (106,'Asha','Pune','asha@x.com','Mouse',600),
 (102,'Ravi','Mumbai','ravi@x.com','Mouse',600),
 (120,'Tom','Pune','tom@x.com','Pen Pack',120);
""")
# 1. update anomaly: the careless UPDATE touches only the order we were looking at
con.execute("UPDATE sales SET customer_city = 'Mumbai' WHERE order_id = 101")
print("update:", con.execute("""SELECT DISTINCT customer_name, customer_city
                                FROM sales WHERE customer_name = 'Asha'""").fetchall())
# 2. insert anomaly: a customer with no order cannot be stored
try:
    con.execute("INSERT INTO sales(customer_name, customer_city, customer_email) VALUES ('Zoya','Delhi','z@x.com')")
except sqlite3.Error as e:
    print("insert:", e)
# 3. delete anomaly: removing Tom's only order removes Tom
con.execute("DELETE FROM sales WHERE order_id = 120")
print("delete:", con.execute("SELECT COUNT(*) FROM sales WHERE customer_name = 'Tom'").fetchone()[0], "rows left for Tom")

update: [('Asha', 'Mumbai'), ('Asha', 'Pune')]
insert: NOT NULL constraint failed: sales.product_name
delete: 0 rows left for Tom


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE sales (order_id INTEGER PRIMARY KEY, customer_name TEXT, customer_city TEXT,
                    customer_email TEXT, product_name TEXT, price INTEGER);
INSERT INTO sales VALUES
 (101,'Asha','Pune','asha@x.com','Keyboard',1500),
 (103,'Asha','Pune','asha@x.com','Desk Lamp',900),
 (106,'Asha','Pune','asha@x.com','Mouse',600),
 (102,'Ravi','Mumbai','ravi@x.com','Mouse',600),
 (120,'Tom','Pune','tom@x.com','Pen Pack',120);

CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT NOT NULL,
                        city TEXT, email TEXT NOT NULL UNIQUE);
INSERT INTO customers(name, city, email)
  SELECT DISTINCT customer_name, customer_city, customer_email FROM sales;

CREATE TABLE products (product_id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, price INTEGER NOT NULL);
INSERT INTO products(name, price) SELECT DISTINCT product_name, price FROM sales;

CREATE TABLE orders (order_id INTEGER PRIMARY KEY,
                     customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
                     product_id  INTEGER NOT NULL REFERENCES products(product_id));
INSERT INTO orders
  SELECT s.order_id, c.customer_id, p.product_id
  FROM sales s
  JOIN customers c ON c.email = s.customer_email
  JOIN products  p ON p.name  = s.product_name;
""")
for t in ("sales", "customers", "products", "orders"):
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])
original = con.execute("SELECT order_id, customer_name, customer_city, customer_email, product_name, price FROM sales").fetchall()
rebuilt = con.execute("""SELECT o.order_id, c.name, c.city, c.email, p.name, p.price
                         FROM orders o JOIN customers c USING (customer_id)
                                       JOIN products  p USING (product_id)""").fetchall()
print("lossless:", sorted(original) == sorted(rebuilt))

sales 5
customers 3
products 4
orders 5
lossless: True


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE people (name TEXT, skills TEXT);
INSERT INTO people VALUES ('Asha','Java, SQL'),('Meena','NoSQL, Python'),('Ravi','Excel');
CREATE TABLE person_skill (name TEXT, skill TEXT, PRIMARY KEY (name, skill));
INSERT INTO person_skill VALUES ('Asha','Java'),('Asha','SQL'),('Meena','NoSQL'),('Meena','Python'),('Ravi','Excel');
""")
print("LIKE on the list :", [r[0] for r in con.execute("SELECT name FROM people WHERE skills LIKE '%SQL%'")])
print("bridge, equality :", [r[0] for r in con.execute("SELECT name FROM person_skill WHERE skill = 'SQL'")])

LIKE on the list : ['Asha', 'Meena']
bridge, equality : ['Asha']


## 1.9 Reading an Unfamiliar Database: Introspection, Counts and Orphans

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT);
CREATE TABLE products  (product_id INTEGER PRIMARY KEY, name TEXT NOT NULL, price INTEGER NOT NULL);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY,
                     customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
                     order_date TEXT NOT NULL, status TEXT NOT NULL, coupon TEXT);
CREATE TABLE order_items (order_id INTEGER NOT NULL REFERENCES orders(order_id),
                          product_id INTEGER NOT NULL REFERENCES products(product_id),
                          qty INTEGER NOT NULL, PRIMARY KEY (order_id, product_id));
""")
print("tables:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")])
print("orders columns (cid, name, type, notnull, default, pk):")
for row in con.execute("PRAGMA table_info(orders)"):
    print("  ", row)
print("order_items foreign keys (from column -> parent table.column):")
for row in con.execute("PRAGMA foreign_key_list(order_items)"):
    print("  ", row[3], "->", f"{row[2]}.{row[4]}")

tables: ['customers', 'order_items', 'orders', 'products']
orders columns (cid, name, type, notnull, default, pk):
   (0, 'order_id', 'INTEGER', 0, None, 1)
   (1, 'customer_id', 'INTEGER', 1, None, 0)
   (2, 'order_date', 'TEXT', 1, None, 0)
   (3, 'status', 'TEXT', 1, None, 0)
   (4, 'coupon', 'TEXT', 0, None, 0)
order_items foreign keys (from column -> parent table.column):
   product_id -> products.product_id
   order_id -> orders.order_id


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, coupon TEXT);
INSERT INTO customers VALUES (1,'Asha'),(2,'Ravi'),(3,'Meena');
INSERT INTO orders VALUES (101,1,NULL),(102,2,'SAVE10'),(103,1,NULL),(104,3,NULL),(105,2,'SAVE10');
""")
names = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
for n in names:
    print(n, con.execute(f"SELECT COUNT(*) FROM {n}").fetchone()[0], "rows")
total, with_coupon = con.execute("SELECT COUNT(*), COUNT(coupon) FROM orders").fetchone()
print("orders:", total, "rows,", with_coupon, "with a coupon,", total - with_coupon, "NULL")

customers 3 rows
orders 5 rows
orders: 5 rows, 2 with a coupon, 3 NULL


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")          # foreign_keys is OFF by default
con.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY,
                     customer_id INTEGER REFERENCES customers(customer_id));
INSERT INTO customers VALUES (1,'Asha'),(2,'Ravi');
INSERT INTO orders VALUES (101,1),(102,2),(103,9),(104,1),(105,7);
""")
print("pragma  :", con.execute("PRAGMA foreign_key_check").fetchall())
anti = con.execute("""SELECT o.order_id, o.customer_id
                      FROM orders o LEFT JOIN customers c ON c.customer_id = o.customer_id
                      WHERE c.customer_id IS NULL ORDER BY o.order_id""").fetchall()
print("anti-join:", anti)

pragma  : [('orders', 103, 'customers', 0), ('orders', 105, 'customers', 0)]
anti-join: [(103, 9), (105, 7)]


## 1.10 Using SQLite from Python: Connections, Parameters and Transactions

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE users (name TEXT, secret TEXT);
INSERT INTO users VALUES ('asha','pw-1'),('ravi','pw-2'),('meena','pw-3');
""")
typed = "x' OR '1'='1"                      # what the attacker typed

unsafe_sql = f"SELECT name FROM users WHERE name = '{typed}'"
print(unsafe_sql)
print("unsafe:", con.execute(unsafe_sql).fetchall())

safe = con.execute("SELECT name FROM users WHERE name = ?", (typed,)).fetchall()
print("safe  :", safe)

SELECT name FROM users WHERE name = 'x' OR '1'='1'
unsafe: [('asha',), ('ravi',), ('meena',)]
safe  : []


In [ ]:
import sqlite3

def balances(con):
    return dict(con.execute("SELECT id, balance FROM accounts ORDER BY id"))

def setup():
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE accounts (id TEXT PRIMARY KEY, balance INTEGER NOT NULL CHECK (balance >= 0))")
    con.executemany("INSERT INTO accounts VALUES (?,?)", [("A", 1000), ("B", 200)])
    con.commit()
    return con

def transfer(con, amount, fail_midway):
    con.execute("UPDATE accounts SET balance = balance - ? WHERE id = 'A'", (amount,))
    if fail_midway:
        raise RuntimeError("network dropped before the credit")
    con.execute("UPDATE accounts SET balance = balance + ? WHERE id = 'B'", (amount,))

# 1. no transaction handling: the debit sticks, the credit never happens
con = setup()
try:
    transfer(con, 500, fail_midway=True)
except RuntimeError as e:
    print("error:", e)
print("without protection:", balances(con), "total =", sum(balances(con).values()))

# 2. with a transaction: the failure undoes the debit
con = setup()
try:
    with con:
        transfer(con, 500, fail_midway=True)
except RuntimeError:
    pass
print("with transaction  :", balances(con), "total =", sum(balances(con).values()))

# 3. the CHECK constraint also protects: overdraw A
con = setup()
try:
    with con:
        transfer(con, 5000, fail_midway=False)
except sqlite3.Error as e:
    print("overdraw refused:", e)
print("after overdraw    :", balances(con))

error: network dropped before the credit
without protection: {'A': 500, 'B': 200} total = 700
with transaction  : {'A': 1000, 'B': 200} total = 1200
overdraw refused: CHECK constraint failed: balance >= 0
after overdraw    : {'A': 1000, 'B': 200}


In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE employees (emp_id INTEGER PRIMARY KEY, name TEXT, dept_id INTEGER, salary INTEGER);
INSERT INTO employees VALUES (1,'Asha',10,90000),(2,'Ravi',20,120000),(3,'Meena',10,95000);
""")
con.row_factory = sqlite3.Row
row = con.execute("SELECT name, salary FROM employees WHERE emp_id = ?", (2,)).fetchone()
print(row["name"], row["salary"], tuple(row))

df = pd.read_sql_query("SELECT dept_id, salary FROM employees WHERE salary >= ?", con, params=(95000,))
print(df)
print(df.groupby("dept_id")["salary"].mean())

Ravi 120000 ('Ravi', 120000)
   dept_id  salary
0       20  120000
1       10   95000
dept_id
10     95000.0
20    120000.0
Name: salary, dtype: float64


## 1.11 How to solve any databases and the relational model problem

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE page_views (user_id INT, page TEXT, viewed_at TEXT);
INSERT INTO page_views VALUES
 (1,'/home','2024-05-01 10:00:00'),(1,'/home','2024-05-01 10:05:00'),
 (1,'/cart','2024-05-01 10:06:00'),(2,'/home','2024-05-01 10:00:00'),
 (2,'/home','2024-05-01 10:00:00');
""")
def test(cols):
    n  = con.execute("SELECT COUNT(*) FROM page_views").fetchone()[0]
    nd = con.execute(f"SELECT COUNT(*) FROM (SELECT DISTINCT {cols} FROM page_views)").fetchone()[0]
    return n, nd, "KEY" if n == nd else "not a key"
print("user_id, page           :", test("user_id, page"))
print("user_id, page, viewed_at:", test("user_id, page, viewed_at"))

user_id, page           : (5, 3, 'not a key')
user_id, page, viewed_at: (5, 4, 'not a key')


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE specialties (specialty_id INTEGER PRIMARY KEY, label TEXT NOT NULL UNIQUE);
CREATE TABLE doctors (doctor_id INTEGER PRIMARY KEY, name TEXT NOT NULL,
                      specialty_id INTEGER NOT NULL REFERENCES specialties(specialty_id));
CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, name TEXT NOT NULL, phone TEXT UNIQUE);
CREATE TABLE appointments (
  appt_id    INTEGER PRIMARY KEY,
  patient_id INTEGER NOT NULL REFERENCES patients(patient_id),
  doctor_id  INTEGER NOT NULL REFERENCES doctors(doctor_id),
  starts_at  TEXT    NOT NULL,
  status     TEXT    NOT NULL DEFAULT 'booked' CHECK (status IN ('booked','done','cancelled')),
  notes      TEXT,
  UNIQUE (doctor_id, starts_at)
);
INSERT INTO specialties VALUES (1,'Cardiology'),(2,'Dermatology');
INSERT INTO doctors VALUES (1,'Dr Rao',1),(2,'Dr Iyer',2);
INSERT INTO patients VALUES (1,'Asha','9000000001'),(2,'Ravi','9000000002');
""")
tests = [
 ("T1 Asha with Dr Rao at 10:00",     (1, 1, "2024-06-01 10:00", "booked")),
 ("T2 Ravi with Dr Rao at 10:00",     (2, 1, "2024-06-01 10:00", "booked")),
 ("T3 Ravi with Dr Iyer at 10:00",    (2, 2, "2024-06-01 10:00", "booked")),
 ("T4 status 'maybe'",                (2, 2, "2024-06-02 09:00", "maybe")),
 ("T5 patient 9 does not exist",      (9, 2, "2024-06-03 09:00", "booked")),
]
for label, args in tests:
    try:
        con.execute("INSERT INTO appointments(patient_id,doctor_id,starts_at,status) VALUES (?,?,?,?)", args)
        print("ok      ", label)
    except sqlite3.Error as e:
        print("REFUSED ", label, "|", e)
print(con.execute("SELECT appt_id, patient_id, doctor_id, status FROM appointments").fetchall())

ok       T1 Asha with Dr Rao at 10:00
REFUSED  T2 Ravi with Dr Rao at 10:00 | UNIQUE constraint failed: appointments.doctor_id, appointments.starts_at
ok       T3 Ravi with Dr Iyer at 10:00
REFUSED  T4 status 'maybe' | CHECK constraint failed: status IN ('booked','done','cancelled')
REFUSED  T5 patient 9 does not exist | FOREIGN KEY constraint failed
[(1, 1, 1, 'booked'), (2, 2, 2, 'booked')]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE ratings (
  user_id  INTEGER NOT NULL,
  movie_id INTEGER NOT NULL,
  stars    INTEGER NOT NULL CHECK (stars BETWEEN 1 AND 5),
  comment  TEXT,
  rated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
  PRIMARY KEY (user_id, movie_id)
);
""")
for args in [(1, 10, 5, None), (1, 10, 4, "changed my mind"), (1, 11, 6, None), (2, 10, 3, "ok")]:
    try:
        con.execute("INSERT INTO ratings(user_id, movie_id, stars, comment) VALUES (?,?,?,?)", args)
        print("accepted", args)
    except sqlite3.Error as e:
        print("refused ", args, "|", e)
print(con.execute("SELECT user_id, movie_id, stars, comment, rated_at IS NOT NULL FROM ratings").fetchall())

accepted (1, 10, 5, None)
refused  (1, 10, 4, 'changed my mind') | UNIQUE constraint failed: ratings.user_id, ratings.movie_id
refused  (1, 11, 6, None) | CHECK constraint failed: stars BETWEEN 1 AND 5
accepted (2, 10, 3, 'ok')
[(1, 10, 5, None, 1), (2, 10, 3, 'ok', 1)]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT NOT NULL UNIQUE);
CREATE TABLE employees (
  emp_id  INTEGER PRIMARY KEY, name TEXT,
  dept_id INTEGER REFERENCES departments(dept_id) ON DELETE CASCADE,
  salary  INTEGER CHECK (salary > 0));
INSERT INTO departments VALUES (10,'Testing'),(20,'Data');
INSERT INTO employees VALUES (1,'Asha',10,90000),(2,'Ravi',20,120000);
""")
steps = [
 ("S1", "INSERT INTO employees VALUES (3,'Tom',30,50000)"),
 ("S2", "UPDATE employees SET salary = -5 WHERE name = 'Asha'"),
 ("S3", "DELETE FROM departments WHERE dept_id = 20"),
 ("S4", "INSERT INTO departments VALUES (50,'Data')"),
 ("S5", "UPDATE departments SET dept_id = 11 WHERE dept_id = 10"),
 ("S6", "DELETE FROM departments WHERE dept_id = 99"),
]
for label, sql in steps:
    try:
        cur = con.execute(sql)
        print(label, "accepted, rows changed:", cur.rowcount)
    except sqlite3.Error as e:
        print(label, "refused:", e)
print(con.execute("SELECT name FROM employees").fetchall(), con.execute("SELECT dept_id, dept_name FROM departments ORDER BY 1").fetchall())

S1 refused: FOREIGN KEY constraint failed
S2 refused: CHECK constraint failed: salary > 0
S3 accepted, rows changed: 1
S4 accepted, rows changed: 1
S5 refused: FOREIGN KEY constraint failed
S6 accepted, rows changed: 0
[('Asha',)] [(10, 'Testing'), (50, 'Data')]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER REFERENCES customers(customer_id), amount INTEGER);
INSERT INTO customers VALUES (1,'Asha','Pune'),(2,'Ravi',NULL),(3,'Meena',NULL);
INSERT INTO orders VALUES (101,1,500),(102,2,NULL),(103,9,300);
""")
def audit(con):
    orphans = {}
    for child, rowid, parent, fk in con.execute("PRAGMA foreign_key_check"):
        orphans[child] = orphans.get(child, 0) + 1
    for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall():
        n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        cols = [r[1] for r in con.execute(f"PRAGMA table_info({t})")]
        nulls = {}
        for c in cols:
            k = n - con.execute(f"SELECT COUNT({c}) FROM {t}").fetchone()[0]
            if k: nulls[c] = k
        print(f"{t}: rows={n}, null counts={nulls}, orphans={orphans.get(t, 0)}")
audit(con)

customers: rows=3, null counts={'city': 2}, orphans=0
orders: rows=3, null counts={'amount': 1}, orphans=1


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE pincodes (pincode TEXT PRIMARY KEY, city TEXT NOT NULL, state TEXT NOT NULL);
CREATE TABLE students (
  student_id INTEGER PRIMARY KEY, name TEXT NOT NULL,
  pincode TEXT REFERENCES pincodes(pincode), fee_paise INTEGER NOT NULL CHECK (fee_paise >= 0));
CREATE TABLE student_phones (
  student_id INTEGER NOT NULL REFERENCES students(student_id),
  phone TEXT NOT NULL, PRIMARY KEY (student_id, phone));
INSERT INTO pincodes VALUES ('411001','Pune','Maharashtra'),('560001','Bengaluru','Karnataka');
INSERT INTO students VALUES (1,'Asha','411001',250000),(2,'Ravi','411001',250000),(3,'Meena','560001',300000);
INSERT INTO student_phones VALUES (1,'9000000001'),(1,'9000000002'),(3,'9000000003');
""")
con.execute("UPDATE pincodes SET city = 'Pune City' WHERE pincode = '411001'")   # ONE edit
print(con.execute("""SELECT s.name, p.city, p.state, s.fee_paise / 100.0 AS fee
                     FROM students s JOIN pincodes p ON p.pincode = s.pincode ORDER BY s.student_id""").fetchall())
print(con.execute("""SELECT s.name, COUNT(sp.phone) FROM students s
                     LEFT JOIN student_phones sp ON sp.student_id = s.student_id
                     GROUP BY s.student_id ORDER BY s.student_id""").fetchall())

[('Asha', 'Pune City', 'Maharashtra', 2500.0), ('Ravi', 'Pune City', 'Maharashtra', 2500.0), ('Meena', 'Bengaluru', 'Karnataka', 3000.0)]
[('Asha', 2), ('Ravi', 0), ('Meena', 1)]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL);
CREATE TABLE order_items (order_id INTEGER, product_id INTEGER, PRIMARY KEY (order_id, product_id));
""")
con.executemany("INSERT INTO customers VALUES (?)", [(i,) for i in range(1, 9)])
con.executemany("INSERT INTO orders VALUES (?,?)", [(101,1),(102,2),(103,1),(104,3),(105,4),(106,1),(107,5),
    (108,2),(109,5),(110,6),(111,1),(112,4),(113,5),(114,2)])
con.executemany("INSERT INTO order_items VALUES (?,?)", [(101,1),(101,3),(102,2),(103,5),(103,4),(104,3),
    (105,1),(106,2),(106,3),(107,4),(108,5),(109,1),(110,3),(110,4),(111,2),(112,5),(113,3),(113,4),(114,1),(114,2)])
q = {
 "a inner  customers-orders        ": "SELECT COUNT(*) FROM customers c JOIN orders o ON o.customer_id = c.customer_id",
 "b left   customers-orders        ": "SELECT COUNT(*) FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id",
 "c left   ... then order_items    ": """SELECT COUNT(*) FROM customers c
        LEFT JOIN orders o ON o.customer_id = c.customer_id
        LEFT JOIN order_items i ON i.order_id = o.order_id""",
}
for name, sql in q.items():
    print(name, con.execute(sql).fetchone()[0])

a inner  customers-orders         14
b left   customers-orders         16
c left   ... then order_items     22


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.execute("CREATE TABLE p (a INTEGER, b TEXT)")
con.execute("CREATE TABLE c (x INTEGER REFERENCES p(a))")
for sql in ["SELECT * FROM nothere", "SELECT zzz FROM p", "CREATE TABLE p (a)",
            "SELEC * FROM p", "INSERT INTO p VALUES (1)", "INSERT INTO c VALUES (1)"]:
    try:
        con.execute(sql)
    except sqlite3.Error as e:
        print(f"{sql:28} -> {e}")

SELECT * FROM nothere        -> no such table: nothere
SELECT zzz FROM p            -> no such column: zzz
CREATE TABLE p (a)           -> table p already exists
SELEC * FROM p               -> near "SELEC": syntax error
INSERT INTO p VALUES (1)     -> table p has 2 columns but 1 values were supplied
INSERT INTO c VALUES (1)     -> foreign key mismatch - "c" referencing "p"


## 1.12 Going Deeper: How a Database Stores, Finds and Protects Rows

You can skip this section on a first read and return after the practice set. It is what separates someone who writes queries from someone who can explain why a query is slow, why a constraint behaved oddly, and what an interviewer is really testing.

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE ev (id INTEGER PRIMARY KEY, user_id INTEGER, note TEXT)")
con.executemany("INSERT INTO ev VALUES (?,?,?)",
                ((i, (i * 7919) % 50000, "x" * 20) for i in range(1, 100001)))
con.commit()
print("page size:", con.execute("PRAGMA page_size").fetchone()[0], "bytes")
print("pages used:", con.execute("PRAGMA page_count").fetchone()[0])

def work(sql, arg):
    n = [0]
    def tick():
        n[0] += 1
        return 0
    con.set_progress_handler(tick, 1)
    rows = con.execute(sql, (arg,)).fetchall()
    con.set_progress_handler(None, 1)
    plan = con.execute("EXPLAIN QUERY PLAN " + sql, (arg,)).fetchone()[3]
    return plan, len(rows), n[0]

print(work("SELECT * FROM ev WHERE id = ?", 777))
print(work("SELECT * FROM ev WHERE user_id = ?", 777))
con.execute("CREATE INDEX ev_user ON ev(user_id)")
print(work("SELECT * FROM ev WHERE user_id = ?", 777))

page size: 4096 bytes
pages used: 794
('SEARCH ev USING INTEGER PRIMARY KEY (rowid=?)', 1, 11)
('SCAN ev', 2, 300016)
('SEARCH ev USING INDEX ev_user (user_id=?)', 2, 25)


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so PRAGMA works
con.execute("PRAGMA foreign_keys = ON")
print("foreign_keys is", con.execute("PRAGMA foreign_keys").fetchone()[0])

def attempt(label, sql, args=()):
    try:
        con.execute(sql, args); print(f"{label:34} accepted")
    except sqlite3.Error as e:
        print(f"{label:34} refused: {e}")

con.execute("CREATE TABLE u (email TEXT UNIQUE)")
attempt("E1 first NULL", "INSERT INTO u VALUES (NULL)")
attempt("E1 second NULL", "INSERT INTO u VALUES (NULL)")
attempt("E2 empty string", "INSERT INTO u VALUES ('')")
attempt("E2 empty string again", "INSERT INTO u VALUES ('')")
print("E2 '' IS NULL ->", con.execute("SELECT '' IS NULL").fetchone()[0])
attempt("E3 asha@x.com", "INSERT INTO u VALUES ('asha@x.com')")
attempt("E3 ASHA@x.com (other case)", "INSERT INTO u VALUES ('ASHA@x.com')")
con.execute("CREATE TABLE u2 (email TEXT UNIQUE COLLATE NOCASE)")
attempt("E3 nocase first", "INSERT INTO u2 VALUES ('asha@x.com')")
attempt("E3 nocase other case", "INSERT INTO u2 VALUES ('ASHA@x.com')")

con.executescript("""
CREATE TABLE par (a INT, b INT, PRIMARY KEY (a, b));
CREATE TABLE chi (a INT, b INT, FOREIGN KEY (a, b) REFERENCES par (a, b));
INSERT INTO par VALUES (1, 1);
""")
attempt("E4 child (1,1) parent exists", "INSERT INTO chi VALUES (1, 1)")
attempt("E4 child (2,2) no parent", "INSERT INTO chi VALUES (2, 2)")
attempt("E4 child (2,NULL) no parent", "INSERT INTO chi VALUES (2, NULL)")

foreign_keys is 1
E1 first NULL                      accepted
E1 second NULL                     accepted
E2 empty string                    accepted
E2 empty string again              refused: UNIQUE constraint failed: u.email
E2 '' IS NULL -> 0
E3 asha@x.com                      accepted
E3 ASHA@x.com (other case)         accepted
E3 nocase first                    accepted
E3 nocase other case               refused: UNIQUE constraint failed: u2.email
E4 child (1,1) parent exists       accepted
E4 child (2,2) no parent           refused: FOREIGN KEY constraint failed
E4 child (2,NULL) no parent        accepted


## 1.13 Mixed-Practice Drill and Self-Test

The practice set at the end of the chapter goes from easy to expert on one topic at a time. This section does the opposite: it mixes everything, the way an interview or a real task does. Try each item on paper first. The answers follow right after each one.

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE a (id INTEGER PRIMARY KEY);
CREATE TABLE b (id INTEGER PRIMARY KEY);
CREATE TABLE t (x INTEGER);
""")
con.executemany("INSERT INTO a VALUES (?)", [(i,) for i in range(1, 5)])
con.executemany("INSERT INTO b VALUES (?)", [(i,) for i in range(1, 7)])
con.executemany("INSERT INTO t VALUES (?)", [(1,), (None,), (3,)])
print("cross join rows       :", con.execute("SELECT COUNT(*) FROM a CROSS JOIN b").fetchone()[0])
print("inner join on b's key :", con.execute("SELECT COUNT(*) FROM a JOIN b ON b.id = a.id").fetchone()[0])
print("COUNT(*), COUNT(x)    :", con.execute("SELECT COUNT(*), COUNT(x) FROM t").fetchone())

cross join rows       : 24
inner join on b's key : 4
COUNT(*), COUNT(x)    : (3, 2)


## 1.14 Where This Chapter Fits: Before and After

This is the first chapter, so nothing comes before it except curiosity. What it gives to the chapters after it:

## Chapter recap

## Practice set

Problem 1.P.1 (easy) Name the primary key of `order_items` in the shop design. **Answer:** The pair `(order_id, product_id)`, a composite key.

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE t (a INTEGER UNIQUE)")
con.executemany("INSERT INTO t VALUES (?)", [(None,), (None,), (1,)])
print(con.execute("SELECT COUNT(*), COUNT(a), COUNT(DISTINCT a) FROM t").fetchone())

(3, 1, 1)


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
print(con.execute("SELECT typeof(1), typeof(1.0), typeof('1'), typeof(NULL), 1/2, 1/2.0").fetchone())

('integer', 'real', 'text', 'null', 0, 0.5)


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE order_items (order_id INTEGER, product_id INTEGER, qty INTEGER, PRIMARY KEY (order_id))")
con.execute("INSERT INTO order_items VALUES (101, 1, 1)")
try:
    con.execute("INSERT INTO order_items VALUES (101, 3, 2)")
except sqlite3.Error as e:
    print("bug:", e)
con.execute("CREATE TABLE order_items2 (order_id INTEGER, product_id INTEGER, qty INTEGER, PRIMARY KEY (order_id, product_id))")
con.execute("INSERT INTO order_items2 VALUES (101, 1, 1)")
con.execute("INSERT INTO order_items2 VALUES (101, 3, 2)")
print("fixed rows:", con.execute("SELECT COUNT(*) FROM order_items2").fetchone()[0])

bug: UNIQUE constraint failed: order_items.order_id
fixed rows: 2


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE room_nights (
  room_id    INTEGER NOT NULL,
  night      TEXT    NOT NULL,
  booking_id INTEGER NOT NULL,
  PRIMARY KEY (room_id, night)
);
INSERT INTO room_nights VALUES (101,'2024-07-01',1),(101,'2024-07-02',1),(101,'2024-07-03',1);
""")
try:
    con.execute("INSERT INTO room_nights VALUES (101,'2024-07-03',2)")   # overlaps the last night
except sqlite3.Error as e:
    print("overlap refused:", e)
con.execute("INSERT INTO room_nights VALUES (101,'2024-07-04',2)")       # next free night is fine
print(con.execute("SELECT booking_id, COUNT(*) FROM room_nights GROUP BY booking_id").fetchall())

overlap refused: UNIQUE constraint failed: room_nights.room_id, room_nights.night
[(1, 3), (2, 1)]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE posts (post_id INTEGER PRIMARY KEY, title TEXT NOT NULL);
CREATE TABLE tags  (tag_id  INTEGER PRIMARY KEY, label TEXT NOT NULL UNIQUE);
CREATE TABLE post_tags (
  post_id INTEGER NOT NULL REFERENCES posts(post_id) ON DELETE CASCADE,
  tag_id  INTEGER NOT NULL REFERENCES tags(tag_id)   ON DELETE CASCADE,
  PRIMARY KEY (post_id, tag_id)
);
INSERT INTO posts VALUES (1,'Intro to SQL'),(2,'Keys explained');
INSERT INTO tags  VALUES (1,'sql'),(2,'beginner'),(3,'design');
INSERT INTO post_tags VALUES (1,1),(1,2),(2,1),(2,3);
""")
for row in con.execute("""SELECT p.title, t.label FROM posts p
                          JOIN post_tags pt ON pt.post_id = p.post_id
                          JOIN tags t ON t.tag_id = pt.tag_id
                          ORDER BY p.post_id, t.label"""):
    print(row)

('Intro to SQL', 'beginner')
('Intro to SQL', 'sql')
('Keys explained', 'design')
('Keys explained', 'sql')


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER);
CREATE TABLE order_items (order_id INTEGER, line INTEGER, PRIMARY KEY (order_id, line));
INSERT INTO customers VALUES (1),(2),(3);
INSERT INTO orders VALUES (1,1),(2,1),(3,3),(4,3),(5,3);
INSERT INTO order_items SELECT order_id, 1 FROM orders UNION ALL SELECT order_id, 2 FROM orders;
""")
a = "SELECT COUNT(*) FROM customers c JOIN orders o ON o.customer_id = c.customer_id"
b = "SELECT COUNT(*) FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id"
c = b + " LEFT JOIN order_items i ON i.order_id = o.order_id"
print([con.execute(q).fetchone()[0] for q in (a, b, c)])

[5, 6, 11]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE students    (student_id INTEGER PRIMARY KEY, email TEXT NOT NULL UNIQUE);
CREATE TABLE instructors (instructor_id INTEGER PRIMARY KEY, name TEXT NOT NULL, office TEXT);
CREATE TABLE courses     (course_id INTEGER PRIMARY KEY, title TEXT NOT NULL,
                          instructor_id INTEGER NOT NULL REFERENCES instructors(instructor_id));
CREATE TABLE enrolments  (student_id INTEGER NOT NULL REFERENCES students(student_id),
                          course_id  INTEGER NOT NULL REFERENCES courses(course_id),
                          grade INTEGER, PRIMARY KEY (student_id, course_id));
INSERT INTO students VALUES (1,'asha@x.com'),(2,'ravi@x.com');
INSERT INTO instructors VALUES (1,'Dr Rao','B-12');
INSERT INTO courses VALUES (10,'Databases',1),(20,'Statistics',1);
INSERT INTO enrolments VALUES (1,10,88),(1,20,75),(2,10,91);
""")
print(con.execute("""SELECT s.email, c.title, i.name, i.office, e.grade
                     FROM enrolments e JOIN students s USING (student_id)
                                       JOIN courses c USING (course_id)
                                       JOIN instructors i USING (instructor_id)
                     ORDER BY s.student_id, c.course_id""").fetchall())

[('asha@x.com', 'Databases', 'Dr Rao', 'B-12', 88), ('asha@x.com', 'Statistics', 'Dr Rao', 'B-12', 75), ('ravi@x.com', 'Databases', 'Dr Rao', 'B-12', 91)]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER);
INSERT INTO customers VALUES (1,'Asha'),(2,'Ravi');
INSERT INTO orders VALUES (101,1),(102,9),(103,9),(104,2),(105,7);
""")
print(con.execute("""SELECT o.customer_id AS missing_customer, COUNT(*) AS orphan_orders
                     FROM orders o LEFT JOIN customers c ON c.customer_id = o.customer_id
                     WHERE c.customer_id IS NULL
                     GROUP BY o.customer_id ORDER BY o.customer_id""").fetchall())

[(7, 1), (9, 2)]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("""CREATE TABLE emp (emp_id INTEGER PRIMARY KEY, manager_id INTEGER REFERENCES emp(emp_id),
               CHECK (manager_id IS NULL OR manager_id <> emp_id))""")
con.execute("INSERT INTO emp VALUES (1, NULL)")
for row in [(2, 1), (3, 3)]:
    try:
        con.execute("INSERT INTO emp VALUES (?,?)", row); print(row, "accepted")
    except sqlite3.Error as e:
        print(row, "refused:", e)
con.execute("UPDATE emp SET manager_id = 2 WHERE emp_id = 1")     # 1 -> 2 -> 1 is a cycle
print("cycle created:", con.execute("SELECT emp_id, manager_id FROM emp ORDER BY emp_id").fetchall())

(2, 1) accepted
(3, 3) refused: CHECK constraint failed: manager_id IS NULL OR manager_id <> emp_id
cycle created: [(1, 2), (2, 1)]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, amount INTEGER NOT NULL)")
batch = [(101, 500), (102, 300), (103, 700)]
for run in (1, 2):
    con.executemany("INSERT INTO orders VALUES (?,?) ON CONFLICT (order_id) DO NOTHING", batch)
    print("after run", run, ":", con.execute("SELECT COUNT(*), SUM(amount) FROM orders").fetchone())

after run 1 : (3, 1500)
after run 2 : (3, 1500)


In [ ]:
import math
rows, row_bytes, page_bytes, fanout = 50_000_000, 100, 8192, 400
rows_per_page = page_bytes // row_bytes
table_pages = math.ceil(rows / rows_per_page)
print("rows per page:", rows_per_page, " table pages:", table_pages, " size GB:", round(table_pages * page_bytes / 1e9, 2))
level, nodes = 1, math.ceil(rows / fanout)
print("index level", level, "pages:", nodes)
while nodes > 1:
    level += 1
    nodes = math.ceil(nodes / fanout)
    print("index level", level, "pages:", nodes)
print("page reads for one lookup:", level + 1, "vs full scan:", table_pages)

rows per page: 81  table pages: 617284  size GB: 5.06
index level 1 pages: 125000
index level 2 pages: 313
index level 3 pages: 1
page reads for one lookup: 4 vs full scan: 617284


In [ ]:
import sqlite3, tempfile, os
path = os.path.join(tempfile.mkdtemp(), "race.db")

def run(with_unique):
    setup = sqlite3.connect(path, isolation_level=None)
    setup.execute("DROP TABLE IF EXISTS users")
    setup.execute("CREATE TABLE users (name TEXT" + (" UNIQUE" if with_unique else "") + ")")
    setup.close()
    a = sqlite3.connect(path, isolation_level=None)
    b = sqlite3.connect(path, isolation_level=None)
    exists_a = a.execute("SELECT COUNT(*) FROM users WHERE name='asha'").fetchone()[0]   # A checks
    exists_b = b.execute("SELECT COUNT(*) FROM users WHERE name='asha'").fetchone()[0]   # B checks
    a.execute("INSERT INTO users VALUES ('asha')")                                         # A inserts
    try:
        b.execute("INSERT INTO users VALUES ('asha')")                                     # B inserts
        outcome = "B inserted too"
    except sqlite3.Error as e:
        outcome = "B refused: " + str(e)
    n = a.execute("SELECT COUNT(*) FROM users").fetchone()[0]
    return exists_a, exists_b, outcome, n

print("no constraint  :", run(False))
print("with UNIQUE    :", run(True))

no constraint  : (0, 0, 'B inserted too', 2)
with UNIQUE    : (0, 0, 'B refused: UNIQUE constraint failed: users.name', 1)


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:", isolation_level=None)
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE products (code TEXT PRIMARY KEY);
CREATE TABLE sales (sale_id INTEGER PRIMARY KEY, code TEXT REFERENCES products(code) ON UPDATE CASCADE);
INSERT INTO products VALUES ('KB-1');
INSERT INTO sales VALUES (1,'KB-1'),(2,'KB-1');
""")
con.execute("UPDATE products SET code = 'KB-100' WHERE code = 'KB-1'")
print(con.execute("SELECT sale_id, code FROM sales ORDER BY sale_id").fetchall())

[(1, 'KB-100'), (2, 'KB-100')]


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:", isolation_level=None)
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE riders  (rider_id  INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE drivers (driver_id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE vehicles (
  vehicle_id INTEGER PRIMARY KEY, driver_id INTEGER NOT NULL REFERENCES drivers(driver_id),
  plate TEXT NOT NULL UNIQUE, UNIQUE (driver_id, vehicle_id));
CREATE TABLE trips (
  trip_id INTEGER PRIMARY KEY,
  rider_id INTEGER NOT NULL REFERENCES riders(rider_id),
  driver_id INTEGER NOT NULL, vehicle_id INTEGER NOT NULL,
  started_at TEXT NOT NULL, ended_at TEXT, fare_paise INTEGER NOT NULL CHECK (fare_paise >= 0),
  CHECK (ended_at IS NULL OR ended_at >= started_at),
  FOREIGN KEY (driver_id, vehicle_id) REFERENCES vehicles(driver_id, vehicle_id));
CREATE TABLE payments (
  payment_id INTEGER PRIMARY KEY, trip_id INTEGER NOT NULL UNIQUE REFERENCES trips(trip_id),
  amount_paise INTEGER NOT NULL CHECK (amount_paise >= 0),
  status TEXT NOT NULL CHECK (status IN ('pending','paid','failed')));
INSERT INTO riders VALUES (1,'Asha');
INSERT INTO drivers VALUES (1,'Ravi'),(2,'Meena');
INSERT INTO vehicles VALUES (1,1,'MH12AB1234'),(2,2,'KA01XY9999');
""")
for label, args in [("ok: driver 1 in own vehicle", (1,1,1)),
                    ("bad: driver 1 in driver 2's vehicle", (1,1,2))]:
    try:
        con.execute("INSERT INTO trips(rider_id,driver_id,vehicle_id,started_at,fare_paise) VALUES (?,?,?,'2024-06-01 10:00',25000)", args)
        print(label, "-> accepted")
    except sqlite3.Error as e:
        print(label, "-> refused:", e)

ok: driver 1 in own vehicle -> accepted
bad: driver 1 in driver 2's vehicle -> refused: FOREIGN KEY constraint failed


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/sql/ch01.html